# Download any dataset from microdata.gov.in

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somdeepkundu/mospi-microdata-kit/blob/main/notebooks/01_download_any_dataset.ipynb)

Find a survey, see its files, download them and get a codebook of every variable, without
writing code. Fill in the boxes on the right of each step and press ▶.

**You need** a free account on [microdata.gov.in](https://microdata.gov.in) and an API key from
your profile page there. Most NSS, PLFS and HCES unit-level data are *registered access*: you
agree to MoSPI's terms when you request them, and those terms do not allow you to share the
household or person records. Share tables and results, not the files.

Built by Somdeep Kundu, RuDRA Lab, C-TARA, IIT Bombay ·
[project page](https://somdeepkundu.github.io/farm-income/) ·
[code](https://github.com/somdeepkundu/mospi-microdata-kit)

## 1. Install the toolkit

In [ ]:
# Installs the toolkit in Colab (about 20 seconds). On your own computer, run
#   pip install "mospi-microdata-kit @ git+https://github.com/somdeepkundu/mospi-microdata-kit@v1.0.0"
# once in a terminal instead.
import importlib.util
import logging
import subprocess
import sys

PACKAGE = "mospi-microdata-kit[progress] @ git+https://github.com/somdeepkundu/mospi-microdata-kit@v1.0.0"
if importlib.util.find_spec("mospi_kit") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGE], check=True)

import mospi_kit

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logging.getLogger("urllib3").setLevel(logging.WARNING)
print("mospi-microdata-kit", mospi_kit.__version__)

## 2. Add your API key

In [ ]:
# Your API key is read from Colab's Secrets panel (the key icon on the left), name MOSPI_API_KEY,
# or typed into a hidden box. It is never printed or saved in this notebook.
import os
from getpass import getpass


def get_api_key():
    if os.environ.get("MOSPI_API_KEY"):
        return os.environ["MOSPI_API_KEY"]
    try:
        from google.colab import userdata
        return userdata.get("MOSPI_API_KEY")
    except Exception:
        return getpass("Paste your microdata.gov.in API key (hidden): ")


client = mospi_kit.MicrodataClient(get_api_key())
print("Ready:", client)

## 3. Search the catalogue
Every word must appear in the title or code, in any order. The first search reads the whole
catalogue (about a minute, the portal allows 10 requests a minute); later searches are instant.

In [ ]:
search_for = "situation assessment agricultural"  #@param {type:"string"}

import pandas as pd
pd.set_option("display.max_colwidth", 120)
found = client.search(search_for)
pd.DataFrame([{"idno": d.idno, "title": d.title, "page": d.url} for d in found])

## 4. List a study's files
Copy an **idno** from the table above (the long code, not the number).

In [ ]:
idno = "DDI-IND-MOSPI-NSSO-70Rnd-Sch33-visit2-Jan-Dec2013"  #@param {type:"string"}

files = client.list_files(idno)
pd.DataFrame([{"file": f.name, "size": f.size_text} for f in files])

## 5. Download
`which_files` takes patterns: `STATA*` for the Stata data, `*.pdf` for reports,
`*` for everything. Separate several with commas. Tick **save_to_google_drive** to keep the files
after Colab closes.

In [ ]:
which_files = "STATA*, *.pdf, schedule*"  #@param {type:"string"}
save_to_google_drive = False  #@param {type:"boolean"}

from pathlib import Path
folder = Path("data") / idno
if save_to_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    folder = Path("/content/drive/MyDrive/microdata") / idno

patterns = [p.strip() for p in which_files.split(",") if p.strip()]
paths = client.download_dataset(idno, folder, include=patterns)
for path in paths:
    if path.suffix.lower() == ".zip":
        mospi_kit.extract(path)
print(f"{len(paths)} files in {folder}. Each file's SHA-256 is in {folder / 'MANIFEST.json'}.")

## 6. What is inside
A codebook lists every variable with its label. It holds no data values, so you can share it.

In [ ]:
tables = mospi_kit.list_tables(folder)
book = mospi_kit.codebook(folder)
book.to_csv(folder / "codebook.csv", index=False)
print(f"{len(tables)} tables, {len(book)} variables. Codebook saved to {folder / 'codebook.csv'}")
book.groupby("file").agg(variables=("variable", "size"), rows=("rows", "max"))

## 7. Open one table
Give a few words from its file name, such as `block 5a`.

In [ ]:
table_words = "block 1&2"  #@param {type:"string"}

path = mospi_kit.find_table(folder, table_words)
df = mospi_kit.read_table(path)
print(path.name, df.shape)
pd.DataFrame({"label": df.attrs["variable_labels"]})

### Before you share this notebook
Use **Edit → Clear all outputs**, so no household rows travel with it.

Next: [the worked example](https://colab.research.google.com/github/somdeepkundu/mospi-microdata-kit/blob/main/notebooks/02_nss70_farm_income_by_state.ipynb)
turns one survey into state estimates with error margins, checked against NSSO's report.